# RL Group Project: Starter Notebook
## Clinical Treatment Optimisation: Sepsis ICU Management

**Master in Data Science & Advanced Analytics — Reinforcement Learning Course**

This project is structured in two stages of increasing complexity.

- In **Configuration A**, you will work with a tabular Sepsis MDP, where the state and action spaces are small enough to apply classical RL methods directly. 

- In **Configuration B**, you will move to a continuous-observation ICU environment that is clinically grounded and significantly more challenging. 

Three realistic failure modes are present in Configuration B, each reflecting a real scenario encountered in clinical AI deployments. The first is episodic observation noise, where monitoring equipment occasionally malfunctions. The second is episodic missing observations, representing situations where lab results are simply unavailable for an entire episode. The third is acute clinical events, which are sudden and irreversible patient deteriorations that occur independently of any treatment given.


---
### Group Members
Fill in before submitting:

Name of the Group: Group X

```
Student 1: 
Student 2: 
Student 3: 
Student 4: 
Student 5: 
```


## 0. Setup & Imports


In [ ]:
# Install dependencies (run once)
# !pip install icu-sepsis numpy pandas matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
import os
import sys
from tqdm import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

os.makedirs('plots', exist_ok=True)
PLOTS_DIR = 'plots'

SEED = 42
np.random.seed(SEED)

#  Import constants and env factory from env_setup.py 
from envs.env_setup import (
    ENV_ID, N_STATES, N_ACTIONS, STATE_SURVIVED, STATE_DIED,
    GAMMA, INTENSITY, SOFA_BIAS, LAM,
    make_sepsis_env,
)

print(f'ICU-Sepsis-v2 | States: {N_STATES} | Actions: {N_ACTIONS}')
print(f'Terminal states: {STATE_SURVIVED} (survived, r=+1)  {STATE_DIED} (died, r=0)')
print('Setup complete!')


In [ ]:
# Configuration already loaded from env_setup.py
# env_setup.py defines: SOFA_BIAS=5.0, LAM=0.02, INTENSITY, make_sepsis_env()
print(f'Required config: sofa_bias={SOFA_BIAS}, lam={LAM}')

---
## 1. Explore the Environment

Before writing any algorithm, take time to understand the MDP you are working with. The insights you gain here should inform your report's Methodology section.

`ICU-Sepsis-v2` is a benchmark MDP constructed from real MIMIC-III patient data. Each episode represents the trajectory of one ICU patient. The agent observes a discrete integer state (ranging from 0 to 715) and must select one of 25 treatment actions corresponding to combinations of vasopressor and IV fluid dose levels. The reward signal is sparse: **+1 at survival, 0 at death, and 0 for all intermediate steps**, with a discount factor γ = 1.


In [ ]:
#  Instantiate and inspect the raw environment 
env = make_sepsis_env()
obs, info = env.reset(seed=SEED)

print(f'Observation space : {env.observation_space} discrete integer state')
print(f'Action space      : {env.action_space}')
print(f'Initial state     : {obs}')
print()

#  Extract the full MDP model 
raw = env.unwrapped
P = raw._tx_mat    # shape (716, 25, 716) — P[s,a,s'] = P(s'|s,a)
R_sasp = raw._r_mat                    # (716, 25, 716) — R[s, a, s']
R = (P * R_sasp).sum(axis=2)          # (716, 25)      — E[r | s, a]

print(f'Transition matrix P : {P.shape}  (S x A x S\')')
print(f'Reward matrix R     : {R.shape}  (S x A)')
print(f'Reward range        : [{R.min():.3f}, {R.max():.3f}]')
print()


In [ ]:
#  Random baseline: establish the performance floor 
def run_random_baseline(n_episodes=1000, seed=SEED):
    np.random.seed(seed)
    env_eval = make_sepsis_env()
    returns, lengths = [], []
    for _ in range(n_episodes):
        obs, _ = env_eval.reset(seed=np.random.randint(100_000))
        total_r, steps, done = 0.0, 0, False
        while not done:
            obs, r, te, tr, _ = env_eval.step(env_eval.action_space.sample())
            total_r += r; steps += 1; done = te or tr
        returns.append(total_r)
        lengths.append(steps)
    env_eval.close()
    return np.array(returns), np.array(lengths)


rand_returns, rand_lengths = run_random_baseline()
survival_rate = float(np.mean(rand_returns > 0)) * 100

print(f'Random agent ({len(rand_returns)} episodes):')
print(f'  Mean return    : {np.mean(rand_returns):.4f}')
print(f'  Survival rate  : {survival_rate:.1f}%')
print(f'  Mean ep length : {np.mean(rand_lengths):.1f} steps')
print()
print('All Config A algorithms must beat the random baseline.')


In [ ]:
#  Visualise state visitation and reward structure 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# State visitation under random policy
np.random.seed(SEED)
env_vis = make_sepsis_env()
visited = []
for _ in range(300):
    obs, _ = env_vis.reset(seed=np.random.randint(100_000))
    done = False
    while not done:
        visited.append(int(obs))
        obs, _, te, tr, _ = env_vis.step(env_vis.action_space.sample())
        done = te or tr
env_vis.close()
clinical = [s for s in visited if s not in (STATE_SURVIVED, STATE_DIED)]

axes[0].hist(clinical, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_xlabel('State index (0-713 = clinical)')
axes[0].set_ylabel('Visit count')
axes[0].set_title('State visitation random policy (300 episodes)', fontweight='bold')

# Max achievable reward per state (only survival-adjacent states have r > 0)
axes[1].bar(range(N_STATES), R.max(axis=1), color='tomato', width=1.0, alpha=0.8)
axes[1].set_xlabel('State index')
axes[1].set_ylabel('max_a R(s,a)')
axes[1].set_title('Reward sparsity only terminal transitions carry reward',
                  fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_env_exploration.png', bbox_inches='tight')
plt.show()


---
# Config A — Tabular Methods

With 716 discrete states and 25 actions, the Q-table has shape `(716, 25)`, totalling 17,900 entries. This size is entirely manageable in memory, which is precisely what motivates the use of tabular algorithms here.

We implement three algorithms in Config A:
- **Policy Iteration (PI)**: a model-based Dynamic Programming method that uses the full MDP (transition matrix P and reward matrix R) to find the theoretically optimal policy. Serves as the performance ceiling.
- **SARSA**: an on-policy, model-free TD algorithm. Learns from environment interaction; its updates use the action the agent actually takes next, making it conservative.
- **Q-Learning**: an off-policy, model-free TD algorithm. Also learns from interaction but always updates towards the best possible next action, making it more aggressive.

The natural contrast: **PI** has access to the full model (unfair advantage but gives us the optimal ceiling), while **SARSA and Q-Learning** must discover the policy through trial and error. Within the model-free pair, the **on-policy vs off-policy** distinction maps directly onto clinical conservatism: an agent that accounts for its own exploratory uncertainty (SARSA) vs one that assumes it will always act optimally (Q-Learning).


---
## 2. Policy Iteration — Optimal Benchmark

Policy Iteration is a Dynamic Programming method that alternates between two steps:
1. **Policy Evaluation**: given the current policy, compute V(s) for all states using the Bellman expectation equation iteratively.
2. **Policy Improvement**: for each state, update the policy to the action that maximises the expected value.

Because the full MDP (P and R) is available in Config A, PI finds the exact optimal policy without any interaction with the environment. This makes it the **theoretical ceiling** against which we measure our model-free agents.


In [ ]:
# ── Policy Iteration ──────────────────────────────────────────────────────────

def policy_iteration(P, R, gamma=1.0, theta=1e-8, max_iter=1000):
    """
    Exact Policy Iteration using the full MDP matrices.

    Parameters
    ----------
    P     : np.ndarray (S, A, S') — transition probabilities
    R     : np.ndarray (S, A)     — expected reward per state-action
    gamma : float                 — discount factor
    theta : float                 — convergence threshold for policy evaluation
    max_iter : int                — max policy improvement iterations

    Returns
    -------
    policy      : np.ndarray (S,) — optimal action per state
    V           : np.ndarray (S,) — optimal value function
    n_iter      : int             — number of policy improvement iterations
    delta_history : list          — max delta per evaluation sweep (for plotting)
    """
    n_states, n_actions, _ = P.shape

    # Initialise with a random policy
    policy = np.zeros(n_states, dtype=int)
    V = np.zeros(n_states)
    delta_history = []

    for iteration in range(max_iter):

        # ── Step 1: Policy Evaluation ────────────────────────────────────────
        # Iteratively compute V(s) for all s under the current policy
        while True:
            delta = 0.0
            for s in range(n_states):
                a = policy[s]
                # Bellman expectation: V(s) = R(s,a) + gamma * sum_s' P(s'|s,a)*V(s')
                v_new = R[s, a] + gamma * np.dot(P[s, a], V)
                delta = max(delta, abs(v_new - V[s]))
                V[s] = v_new
            delta_history.append(delta)
            if delta < theta:
                break

        # ── Step 2: Policy Improvement ───────────────────────────────────────
        # For each state, greedily pick the action with highest Q(s,a)
        policy_stable = True
        for s in range(n_states):
            old_action = policy[s]
            # Q(s,a) = R(s,a) + gamma * sum_s' P(s'|s,a)*V(s')
            Q_s = R[s] + gamma * P[s].dot(V)   # shape (A,)
            policy[s] = np.argmax(Q_s)
            if old_action != policy[s]:
                policy_stable = False

        if policy_stable:
            print(f'Policy Iteration converged after {iteration + 1} improvement iterations.')
            break

    return policy, V, iteration + 1, delta_history


# Run Policy Iteration
np.random.seed(SEED)
pi_policy, pi_V, pi_iters, pi_deltas = policy_iteration(P, R, gamma=GAMMA)

print(f'Value function range: [{pi_V.min():.4f}, {pi_V.max():.4f}]')
print(f'Unique actions used by PI policy: {len(np.unique(pi_policy))}/25')


In [ ]:
# ── Evaluate Policy Iteration ─────────────────────────────────────────────────

def evaluate_policy_tabular(policy_array, n_episodes=1000, seed=SEED):
    """
    Evaluate a deterministic tabular policy over n_episodes.
    policy_array : np.ndarray (N_STATES,) — action index per state
    Returns mean return, survival rate, mean episode length.
    """
    np.random.seed(seed)
    env_eval = make_sepsis_env()
    returns, lengths = [], []

    for _ in range(n_episodes):
        obs, _ = env_eval.reset(seed=np.random.randint(100_000))
        total_r, steps, done = 0.0, 0, False
        while not done:
            action = int(policy_array[int(obs)])
            obs, r, te, tr, _ = env_eval.step(action)
            total_r += r
            steps += 1
            done = te or tr
        returns.append(total_r)
        lengths.append(steps)

    env_eval.close()
    returns = np.array(returns)
    return {
        'mean_return'   : float(np.mean(returns)),
        'survival_rate' : float(np.mean(returns > 0)) * 100,
        'mean_length'   : float(np.mean(lengths)),
    }


pi_results = evaluate_policy_tabular(pi_policy)
print('Policy Iteration — Evaluation (1000 episodes):')
print(f'  Mean return   : {pi_results["mean_return"]:.4f}')
print(f'  Survival rate : {pi_results["survival_rate"]:.1f}%')
print(f'  Mean length   : {pi_results["mean_length"]:.1f} steps')
print(f'  vs Random baseline survival: {survival_rate:.1f}%')


In [ ]:
# ── Plot PI: convergence + value function + policy heatmap ───────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 1. Convergence: max delta per evaluation sweep
axes[0].semilogy(pi_deltas, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Evaluation sweep')
axes[0].set_ylabel('Max |V change| (log scale)')
axes[0].set_title('Policy Iteration — Convergence', fontweight='bold')

# 2. Optimal value function across clinical states
clinical_states = [s for s in range(N_STATES) if s not in (STATE_SURVIVED, STATE_DIED)]
axes[1].bar(clinical_states, pi_V[clinical_states], color='tomato', width=1.0, alpha=0.8)
axes[1].set_xlabel('State index')
axes[1].set_ylabel('V*(s)')
axes[1].set_title('Optimal Value Function V*', fontweight='bold')

# 3. Policy heatmap: action chosen per state (25 actions = 5 vaso x 5 fluid)
# Reshape policy into a grid for visualisation
policy_display = pi_policy[:714]   # clinical states only
# Interpret action as (vaso_level, fluid_level): action = vaso*5 + fluid
vaso_levels  = policy_display // 5
fluid_levels = policy_display  % 5

axes[2].scatter(range(len(policy_display)), vaso_levels,
                c=fluid_levels, cmap='RdYlGn', s=4, alpha=0.6)
axes[2].set_xlabel('State index')
axes[2].set_ylabel('Vasopressor level (0=none, 4=high)')
axes[2].set_title('PI Policy: vasopressor level per state\n(colour = IV fluid level)', fontweight='bold')
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(0, 4))
plt.colorbar(sm, ax=axes[2], label='IV fluid level')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_PI.png', bbox_inches='tight')
plt.show()


---
## 3. SARSA — On-Policy TD Control

SARSA (**S**tate–**A**ction–**R**eward–next **S**tate–next **A**ction) is a model-free, on-policy TD control algorithm. At each step it uses the sequence `(s, a, r, s', a')` to update the Q-value:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[r + \gamma\, Q(s', a') - Q(s,a)\bigr]$$

The key: **a' is selected by the same epsilon-greedy policy** the agent is currently following. This means SARSA's updates account for the risk of future exploratory (random) actions — making it naturally more conservative than Q-Learning. In a clinical context this conservatism is desirable: the agent effectively learns a policy that is safe even while it is still uncertain.


In [ ]:
# ── SARSA ─────────────────────────────────────────────────────────────────────

def sarsa(
    n_episodes=50_000,
    alpha=0.3,
    gamma=1.0,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
):
    """
    SARSA on-policy TD control for the tabular Sepsis MDP.

    Parameters
    ----------
    n_episodes    : int   — number of training episodes
    alpha         : float — learning rate
    gamma         : float — discount factor
    epsilon_start : float — initial exploration probability
    epsilon_min   : float — minimum exploration probability
    seed          : int   — random seed

    Returns
    -------
    Q            : np.ndarray (N_STATES, N_ACTIONS) — learned Q-table
    returns_log  : list of float — episode returns during training
    """
    np.random.seed(seed)
    env_train = make_sepsis_env()

    Q = np.zeros((N_STATES, N_ACTIONS))
    returns_log = []
    epsilon = epsilon_start
    decay = (epsilon_start - epsilon_min) / n_episodes

    for ep in tqdm(range(n_episodes), desc='SARSA', leave=False):
        obs, _ = env_train.reset(seed=np.random.randint(100_000))
        s = int(obs)

        # Select first action with epsilon-greedy
        if np.random.random() < epsilon:
            a = env_train.action_space.sample()
        else:
            a = int(np.argmax(Q[s]))

        total_r, done = 0.0, False

        while not done:
            obs_next, r, te, tr, _ = env_train.step(a)
            s_next = int(obs_next)
            done = te or tr

            # Select next action (on-policy: epsilon-greedy)
            if np.random.random() < epsilon:
                a_next = env_train.action_space.sample()
            else:
                a_next = int(np.argmax(Q[s_next]))

            # SARSA update — uses actual next action a_next
            td_target = r + gamma * Q[s_next, a_next] * (not done)
            Q[s, a] += alpha * (td_target - Q[s, a])

            s, a = s_next, a_next
            total_r += r

        returns_log.append(total_r)
        # Decay epsilon after each episode
        epsilon = max(epsilon_min, epsilon - decay)

    env_train.close()
    return Q, returns_log


# Run SARSA
SARSA_N_EPISODES = 50_000
sarsa_Q, sarsa_returns = sarsa(
    n_episodes=SARSA_N_EPISODES,
    alpha=0.3,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_min=0.01,
)

sarsa_policy = np.argmax(sarsa_Q, axis=1)
print('SARSA training complete.')
print(f'Unique actions used: {len(np.unique(sarsa_policy))}/25')


In [ ]:
# ── Evaluate SARSA ────────────────────────────────────────────────────────────
sarsa_results = evaluate_policy_tabular(sarsa_policy)
print('SARSA — Evaluation (1000 episodes):')
print(f'  Mean return   : {sarsa_results["mean_return"]:.4f}')
print(f'  Survival rate : {sarsa_results["survival_rate"]:.1f}%')
print(f'  Mean length   : {sarsa_results["mean_length"]:.1f} steps')


In [ ]:
# ── Plot SARSA: learning curve + Q-value distribution ────────────────────────

window = 500
sarsa_rolling = pd.Series(sarsa_returns).rolling(window).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Learning curve
axes[0].plot(sarsa_returns, alpha=0.15, color='steelblue', linewidth=0.5, label='Episode return')
axes[0].plot(sarsa_rolling, color='steelblue', linewidth=2.0, label=f'Rolling mean ({window})')
axes[0].axhline(pi_results['mean_return'], color='tomato', linestyle='--', linewidth=1.5, label='PI ceiling')
axes[0].axhline(np.mean(rand_returns), color='gray', linestyle=':', linewidth=1.5, label='Random baseline')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Return')
axes[0].set_title('SARSA — Learning Curve', fontweight='bold')
axes[0].legend(fontsize=8)

# Q-value distribution (max Q per state, clinical states only)
max_Q_sarsa = sarsa_Q[:714].max(axis=1)
axes[1].hist(max_Q_sarsa, bins=40, color='steelblue', edgecolor='none', alpha=0.8)
axes[1].set_xlabel('max_a Q(s,a)')
axes[1].set_ylabel('State count')
axes[1].set_title('SARSA — Q-value Distribution (clinical states)', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_SARSA.png', bbox_inches='tight')
plt.show()


---
## 4. Q-Learning — Off-Policy TD Control

Q-Learning is a model-free, **off-policy** TD control algorithm. Its update rule differs from SARSA in one crucial place — instead of using the action the agent actually takes next (`a'`), it always bootstraps from the **maximum** Q-value in the next state:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[r + \gamma \max_{a'} Q(s', a') - Q(s,a)\bigr]$$

This means Q-Learning learns the value of the **greedy policy** regardless of how the agent actually behaves during training. It tends to converge to a more aggressive optimal policy, but may overestimate Q-values because it always assumes the best possible future (maximisation bias).

In the clinical context: Q-Learning may recommend higher-intensity interventions. Comparing its policy to SARSA's will reveal whether aggressive treatment strategies actually improve survival in this dataset.


In [ ]:
# ── Q-Learning ────────────────────────────────────────────────────────────────

def q_learning(
    n_episodes=50_000,
    alpha=0.3,
    gamma=1.0,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
):
    """
    Q-Learning off-policy TD control for the tabular Sepsis MDP.

    Parameters
    ----------
    n_episodes    : int   — number of training episodes
    alpha         : float — learning rate
    gamma         : float — discount factor
    epsilon_start : float — initial exploration probability
    epsilon_min   : float — minimum exploration probability
    seed          : int   — random seed

    Returns
    -------
    Q            : np.ndarray (N_STATES, N_ACTIONS) — learned Q-table
    returns_log  : list of float — episode returns during training
    """
    np.random.seed(seed)
    env_train = make_sepsis_env()

    Q = np.zeros((N_STATES, N_ACTIONS))
    returns_log = []
    epsilon = epsilon_start
    decay = (epsilon_start - epsilon_min) / n_episodes

    for ep in tqdm(range(n_episodes), desc='Q-Learning', leave=False):
        obs, _ = env_train.reset(seed=np.random.randint(100_000))
        s = int(obs)
        total_r, done = 0.0, False

        while not done:
            # Select action with epsilon-greedy
            if np.random.random() < epsilon:
                a = env_train.action_space.sample()
            else:
                a = int(np.argmax(Q[s]))

            obs_next, r, te, tr, _ = env_train.step(a)
            s_next = int(obs_next)
            done = te or tr

            # Q-Learning update — uses max over next actions (off-policy)
            td_target = r + gamma * np.max(Q[s_next]) * (not done)
            Q[s, a] += alpha * (td_target - Q[s, a])

            s = s_next
            total_r += r

        returns_log.append(total_r)
        epsilon = max(epsilon_min, epsilon - decay)

    env_train.close()
    return Q, returns_log


# Run Q-Learning
QL_N_EPISODES = 50_000
ql_Q, ql_returns = q_learning(
    n_episodes=QL_N_EPISODES,
    alpha=0.3,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_min=0.01,
)

ql_policy = np.argmax(ql_Q, axis=1)
print('Q-Learning training complete.')
print(f'Unique actions used: {len(np.unique(ql_policy))}/25')


In [ ]:
# ── Evaluate Q-Learning ───────────────────────────────────────────────────────
ql_results = evaluate_policy_tabular(ql_policy)
print('Q-Learning — Evaluation (1000 episodes):')
print(f'  Mean return   : {ql_results["mean_return"]:.4f}')
print(f'  Survival rate : {ql_results["survival_rate"]:.1f}%')
print(f'  Mean length   : {ql_results["mean_length"]:.1f} steps')


In [ ]:
# ── Plot Q-Learning: learning curve + SARSA vs QL overlay ────────────────────

window = 500
ql_rolling   = pd.Series(ql_returns).rolling(window).mean()
sarsa_rolling_full = pd.Series(sarsa_returns).rolling(window).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Q-Learning learning curve alone
axes[0].plot(ql_returns, alpha=0.15, color='darkorange', linewidth=0.5, label='Episode return')
axes[0].plot(ql_rolling, color='darkorange', linewidth=2.0, label=f'Rolling mean ({window})')
axes[0].axhline(pi_results['mean_return'], color='tomato', linestyle='--', linewidth=1.5, label='PI ceiling')
axes[0].axhline(np.mean(rand_returns), color='gray', linestyle=':', linewidth=1.5, label='Random baseline')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Return')
axes[0].set_title('Q-Learning — Learning Curve', fontweight='bold')
axes[0].legend(fontsize=8)

# Head-to-head: SARSA vs Q-Learning rolling mean
axes[1].plot(sarsa_rolling_full, color='steelblue', linewidth=2.0, label='SARSA')
axes[1].plot(ql_rolling, color='darkorange', linewidth=2.0, label='Q-Learning')
axes[1].axhline(pi_results['mean_return'], color='tomato', linestyle='--', linewidth=1.5, label='PI ceiling')
axes[1].axhline(np.mean(rand_returns), color='gray', linestyle=':', linewidth=1.5, label='Random baseline')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Rolling mean return')
axes[1].set_title('SARSA vs Q-Learning — Comparison', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_QL.png', bbox_inches='tight')
plt.show()


---
## 5. Creative Extension — Double Q-Learning

Standard Q-Learning suffers from **maximisation bias**: because it always bootstraps from `max Q(s', .)`, it systematically overestimates action values when the Q-table contains noise. In a medical context this matters — overestimating the value of an aggressive treatment could lead the agent to over-prescribe vasopressors or fluids even when moderate doses are sufficient.

**Double Q-Learning** (Hasselt, 2010) addresses this by maintaining **two independent Q-tables** (`Q_A` and `Q_B`). At each step:
- With probability 0.5, use `Q_A` to *select* the best next action, but `Q_B` to *evaluate* it — and update `Q_A`.
- Otherwise do the reverse.

This decoupling of action selection from action evaluation removes the upward bias. We expect Double Q-Learning to produce a **more conservative treatment policy** than standard Q-Learning, which is clinically desirable.


In [ ]:
# ── Double Q-Learning ─────────────────────────────────────────────────────────

def double_q_learning(
    n_episodes=50_000,
    alpha=0.3,
    gamma=1.0,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
):
    """
    Double Q-Learning to reduce maximisation bias.

    Maintains two Q-tables (Q_A, Q_B). At each step, randomly chooses
    which table to update, using the other for evaluation.

    Parameters
    ----------
    n_episodes    : int   — number of training episodes
    alpha         : float — learning rate
    gamma         : float — discount factor
    epsilon_start : float — initial exploration probability
    epsilon_min   : float — minimum exploration probability
    seed          : int   — random seed

    Returns
    -------
    Q_avg        : np.ndarray (N_STATES, N_ACTIONS) — average of Q_A and Q_B
    returns_log  : list of float — episode returns during training
    """
    np.random.seed(seed)
    env_train = make_sepsis_env()

    Q_A = np.zeros((N_STATES, N_ACTIONS))
    Q_B = np.zeros((N_STATES, N_ACTIONS))
    returns_log = []
    epsilon = epsilon_start
    decay = (epsilon_start - epsilon_min) / n_episodes

    for ep in tqdm(range(n_episodes), desc='Double Q-Learning', leave=False):
        obs, _ = env_train.reset(seed=np.random.randint(100_000))
        s = int(obs)
        total_r, done = 0.0, False

        while not done:
            # Action selection uses the sum of both tables
            Q_sum = Q_A[s] + Q_B[s]
            if np.random.random() < epsilon:
                a = env_train.action_space.sample()
            else:
                a = int(np.argmax(Q_sum))

            obs_next, r, te, tr, _ = env_train.step(a)
            s_next = int(obs_next)
            done = te or tr

            # Randomly assign update to Q_A or Q_B
            if np.random.random() < 0.5:
                # Update Q_A: select action with Q_A, evaluate with Q_B
                best_a_next = int(np.argmax(Q_A[s_next]))
                td_target = r + gamma * Q_B[s_next, best_a_next] * (not done)
                Q_A[s, a] += alpha * (td_target - Q_A[s, a])
            else:
                # Update Q_B: select action with Q_B, evaluate with Q_A
                best_a_next = int(np.argmax(Q_B[s_next]))
                td_target = r + gamma * Q_A[s_next, best_a_next] * (not done)
                Q_B[s, a] += alpha * (td_target - Q_B[s, a])

            s = s_next
            total_r += r

        returns_log.append(total_r)
        epsilon = max(epsilon_min, epsilon - decay)

    env_train.close()
    Q_avg = (Q_A + Q_B) / 2.0
    return Q_avg, returns_log


# Run Double Q-Learning
dql_Q, dql_returns = double_q_learning(
    n_episodes=50_000,
    alpha=0.3,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_min=0.01,
)

dql_policy = np.argmax(dql_Q, axis=1)
print('Double Q-Learning training complete.')
print(f'Unique actions used: {len(np.unique(dql_policy))}/25')


In [ ]:
# ── Evaluate Double Q-Learning ────────────────────────────────────────────────
dql_results = evaluate_policy_tabular(dql_policy)
print('Double Q-Learning — Evaluation (1000 episodes):')
print(f'  Mean return   : {dql_results["mean_return"]:.4f}')
print(f'  Survival rate : {dql_results["survival_rate"]:.1f}%')
print(f'  Mean length   : {dql_results["mean_length"]:.1f} steps')


In [ ]:
# ── Maximisation bias comparison: Q-Learning vs Double Q-Learning ─────────────
# Show Q-value distributions to reveal bias in standard Q-Learning

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Max Q-value per state — higher values in standard QL suggest overestimation
max_Q_ql  = ql_Q[:714].max(axis=1)
max_Q_dql = dql_Q[:714].max(axis=1)

axes[0].hist(max_Q_ql, bins=40, alpha=0.6, color='darkorange', label='Q-Learning', density=True)
axes[0].hist(max_Q_dql, bins=40, alpha=0.6, color='purple', label='Double Q-Learning', density=True)
axes[0].axvline(max_Q_ql.mean(), color='darkorange', linestyle='--', linewidth=1.5,
                label=f'QL mean = {max_Q_ql.mean():.3f}')
axes[0].axvline(max_Q_dql.mean(), color='purple', linestyle='--', linewidth=1.5,
                label=f'DQL mean = {max_Q_dql.mean():.3f}')
axes[0].set_xlabel('max_a Q(s,a)')
axes[0].set_ylabel('Density')
axes[0].set_title('Q-value Distribution: Bias Check', fontweight='bold')
axes[0].legend(fontsize=8)

# Learning curves: all three model-free methods
dql_rolling = pd.Series(dql_returns).rolling(500).mean()
axes[1].plot(sarsa_rolling_full, color='steelblue', linewidth=2.0, label='SARSA')
axes[1].plot(ql_rolling, color='darkorange', linewidth=2.0, label='Q-Learning')
axes[1].plot(dql_rolling, color='purple', linewidth=2.0, label='Double Q-Learning')
axes[1].axhline(pi_results['mean_return'], color='tomato', linestyle='--', linewidth=1.5, label='PI ceiling')
axes[1].axhline(np.mean(rand_returns), color='gray', linestyle=':', linewidth=1.5, label='Random baseline')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Rolling mean return')
axes[1].set_title('All Model-Free Agents — Learning Curves', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_DQL.png', bbox_inches='tight')
plt.show()


---
## 6. Config A — Final Comparison & Analysis

We now compare all four approaches:
- **Random baseline** — performance floor
- **Policy Iteration** — optimal ceiling (model-based)
- **SARSA** — on-policy, model-free, conservative
- **Q-Learning** — off-policy, model-free, aggressive
- **Double Q-Learning** — off-policy, model-free, bias-corrected


In [ ]:
# ── Summary Results Table ─────────────────────────────────────────────────────

results_table = pd.DataFrame([
    {
        'Algorithm'       : 'Random Baseline',
        'Type'            : 'Baseline',
        'Needs Model'     : 'No',
        'Mean Return'     : round(float(np.mean(rand_returns)), 4),
        'Survival Rate %' : round(survival_rate, 1),
        'Mean Ep Length'  : round(float(np.mean(rand_lengths)), 1),
    },
    {
        'Algorithm'       : 'Policy Iteration',
        'Type'            : 'Dynamic Programming',
        'Needs Model'     : 'Yes',
        'Mean Return'     : round(pi_results['mean_return'], 4),
        'Survival Rate %' : round(pi_results['survival_rate'], 1),
        'Mean Ep Length'  : round(pi_results['mean_length'], 1),
    },
    {
        'Algorithm'       : 'SARSA',
        'Type'            : 'On-Policy TD',
        'Needs Model'     : 'No',
        'Mean Return'     : round(sarsa_results['mean_return'], 4),
        'Survival Rate %' : round(sarsa_results['survival_rate'], 1),
        'Mean Ep Length'  : round(sarsa_results['mean_length'], 1),
    },
    {
        'Algorithm'       : 'Q-Learning',
        'Type'            : 'Off-Policy TD',
        'Needs Model'     : 'No',
        'Mean Return'     : round(ql_results['mean_return'], 4),
        'Survival Rate %' : round(ql_results['survival_rate'], 1),
        'Mean Ep Length'  : round(ql_results['mean_length'], 1),
    },
    {
        'Algorithm'       : 'Double Q-Learning',
        'Type'            : 'Off-Policy TD (bias-corrected)',
        'Needs Model'     : 'No',
        'Mean Return'     : round(dql_results['mean_return'], 4),
        'Survival Rate %' : round(dql_results['survival_rate'], 1),
        'Mean Ep Length'  : round(dql_results['mean_length'], 1),
    },
])

display(results_table)
results_table.to_csv(f'{PLOTS_DIR}/configA_results_table.csv', index=False)


In [ ]:
# ── Bar chart: Survival Rate comparison ──────────────────────────────────────

labels   = results_table['Algorithm']
survival = results_table['Survival Rate %']
colors   = ['gray', 'tomato', 'steelblue', 'darkorange', 'purple']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Survival rate bar chart
bars = axes[0].bar(labels, survival, color=colors, alpha=0.85, edgecolor='white')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('Config A — Survival Rate by Algorithm', fontweight='bold')
axes[0].set_ylim(0, 105)
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, survival):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Policy agreement heatmap: how much do the three model-free policies agree?
policy_matrix = np.stack([
    (pi_policy == sarsa_policy).astype(int),
    (pi_policy == ql_policy).astype(int),
    (pi_policy == dql_policy).astype(int),
    (sarsa_policy == ql_policy).astype(int),
    (sarsa_policy == dql_policy).astype(int),
    (ql_policy == dql_policy).astype(int),
], axis=0)  # shape (6, 716)

agreement_rates = policy_matrix.mean(axis=1) * 100
pairs = ['PI vs SARSA', 'PI vs QL', 'PI vs DQL',
         'SARSA vs QL', 'SARSA vs DQL', 'QL vs DQL']

pair_bars = axes[1].barh(pairs, agreement_rates, color='teal', alpha=0.75)
axes[1].set_xlabel('Policy Agreement (%)')
axes[1].set_title('Policy Agreement Between Algorithms', fontweight='bold')
axes[1].set_xlim(0, 110)
for bar, val in zip(pair_bars, agreement_rates):
    axes[1].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_comparison.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Treatment Intensity Analysis: clinical interpretability ──────────────────
# Compare the distribution of vasopressor and IV fluid doses chosen by each policy

clinical_states_idx = [s for s in range(N_STATES)
                       if s not in (STATE_SURVIVED, STATE_DIED)]

def get_dose_distributions(policy_arr, state_idx):
    """Return vaso and fluid level distributions for clinical states."""
    actions = policy_arr[state_idx]
    vaso  = actions // 5
    fluid = actions  % 5
    return vaso, fluid


fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey='row')
policies_to_plot = [
    ('Policy Iteration', pi_policy,   'tomato'),
    ('SARSA',            sarsa_policy, 'steelblue'),
    ('Q-Learning',       ql_policy,    'darkorange'),
    ('Double Q-Learning',dql_policy,   'purple'),
]

for col, (name, pol, col_color) in enumerate(policies_to_plot):
    vaso, fluid = get_dose_distributions(pol, clinical_states_idx)

    axes[0, col].hist(vaso, bins=np.arange(-0.5, 5.5), color=col_color, alpha=0.8, edgecolor='white')
    axes[0, col].set_title(name, fontweight='bold', fontsize=9)
    axes[0, col].set_xlabel('Vasopressor level')
    if col == 0:
        axes[0, col].set_ylabel('State count')

    axes[1, col].hist(fluid, bins=np.arange(-0.5, 5.5), color=col_color, alpha=0.8, edgecolor='white')
    axes[1, col].set_xlabel('IV Fluid level')
    if col == 0:
        axes[1, col].set_ylabel('State count')

axes[0, 0].set_ylabel('State count\n(Vasopressor)', fontsize=9)
axes[1, 0].set_ylabel('State count\n(IV Fluid)', fontsize=9)

plt.suptitle('Treatment Intensity Distribution per Policy\n(clinical states only)',
             fontweight='bold', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_treatment_intensity.png', bbox_inches='tight')
plt.show()

# Print mean dose levels for each algorithm
print('Mean treatment intensity per algorithm (clinical states):')
print(f'{"Algorithm":<22} {"Mean Vaso":>12} {"Mean Fluid":>12}')
print('-' * 48)
for name, pol, _ in policies_to_plot:
    vaso, fluid = get_dose_distributions(pol, clinical_states_idx)
    print(f'{name:<22} {vaso.mean():>12.3f} {fluid.mean():>12.3f}')


In [ ]:
# ── Final Config A summary print ──────────────────────────────────────────────
print('=' * 60)
print('CONFIG A — FINAL SUMMARY')
print('=' * 60)
print(f'{"Algorithm":<22} {"Survival %":>12} {"Mean Return":>12}')
print('-' * 60)
rows = [
    ('Random Baseline',    survival_rate,                    np.mean(rand_returns)),
    ('Policy Iteration',   pi_results['survival_rate'],      pi_results['mean_return']),
    ('SARSA',              sarsa_results['survival_rate'],   sarsa_results['mean_return']),
    ('Q-Learning',         ql_results['survival_rate'],      ql_results['mean_return']),
    ('Double Q-Learning',  dql_results['survival_rate'],     dql_results['mean_return']),
]
for name, sr, mr in rows:
    print(f'{name:<22} {sr:>11.1f}% {mr:>12.4f}')
print('=' * 60)
print()
print('Key takeaways:')
print('  - PI sets the optimal ceiling (uses full MDP model).')
print('  - SARSA and Q-Learning both beat the random baseline.')
print('  - Double Q-Learning shows reduced maximisation bias')
print('    (lower mean Q-values) and may suggest more conservative')
print('    treatment strategies, relevant in a clinical setting.')


---
# Config B: RL on the Clinical ICU-Sepsis Environment

In Config A, patient state was represented as a discrete integer produced by discretising a set of clinical measurements into a small number of categories. **Config B uses the full ICU-Sepsis environment**, also built from MIMIC-III data, but with two fundamental changes that make the problem substantially harder.

**Change 1 — Continuous observations.**
The agent now receives a **47-dimensional continuous feature vector** instead of a single discrete index. This vector contains the actual normalised physiological measurements used in the original Komorowski et al. (2018) AI Clinician study, including SOFA score, heart rate, lactate, blood pressure, creatinine, and 42 other clinical variables.

With continuous observations, a tabular Q-table is no longer feasible: it would require one entry per unique float vector, making it effectively infinite. 

**Change 2 — Clinical reality wrappers.**
Config B injects three orthogonal failure modes that reflect challenges faced by real clinical AI deployments.

The first wrapper, `EpisodicNoisyObsEnv`, models episodic monitor malfunction. When active, the observations received by the agent are corrupted by noise for the entire episode, testing robustness to measurement error.

The second wrapper, `EpisodicMissingObsEnv`, models situations where lab results are unavailable for a full episode. This tests how well the agent handles partial observability.

The third wrapper, `AcuteEventEnv`, introduces rare, sudden patient deterioration events such as cardiac arrest or acute organ failure. These occur independently of any treatment decision and represent irreducible stochasticity in the environment.


Key environment properties for Config B:
- **Actions**: 25 total (5 vasopressor levels x 5 IV fluid dose levels)
- **Reward**: +1.0 at survival, 0.0 at death, plus a small treatment intensity penalty (lam = 0.02)
- **Observation**: `Box(47,)`, a normalised physiological feature vector, potentially noisy or incomplete

## Setup: Clinical ICU-Sepsis Environment


In [ ]:
#  Import Clinical Reality Wrappers from wrappers.py 

from envs.wrappers import (
    EpisodicNoisyObsEnv,
    EpisodicMissingObsEnv,
    AcuteEventEnv,
    make_clinical_env,
)

print('Clinical reality wrappers imported from wrappers.py:')
print('  EpisodicNoisyObsEnv   : episodic monitor malfunction')
print('  EpisodicMissingObsEnv : episodic missing lab values')
print('  AcuteEventEnv         : rare sudden patient death')
print()
print('Required Config B env: make_clinical_env() with default parameters')


In [ ]:
#  Verify wrappers and random baseline on clinical environment 
import gymnasium as gym

try:
    env_clinical = make_clinical_env()
    obs, info = env_clinical.reset(seed=SEED)

    print('Clinical environment loaded successfully!')
    print(f'Observation space : {env_clinical.observation_space}')
    print(f'Action space      : {env_clinical.action_space}')
    print(f'Info keys         : {list(info.keys())}')
    print()

    # Random baseline on clinical env (1000 episodes)
    np.random.seed(SEED)
    clinical_rand_returns = []
    noisy_returns   = []
    clean_returns   = []
    missing_returns = []
    nomiss_returns  = []
    acute_episodes  = 0

    env_eval = make_clinical_env()
    for ep in range(1000):
        obs, info = env_eval.reset(seed=np.random.randint(100_000))
        total_r, done = 0.0, False
        ep_noisy   = info.get('noisy_episode', False)
        ep_missing = info.get('missing_features') is not None
        ep_acute   = False

        while not done:
            obs, r, te, tr, info = env_eval.step(env_eval.action_space.sample())
            total_r += r
            done = te or tr
            if info.get('acute_event', False):
                ep_acute = True

        clinical_rand_returns.append(total_r)
        if ep_noisy:   noisy_returns.append(total_r)
        else:          clean_returns.append(total_r)
        if ep_missing: missing_returns.append(total_r)
        else:          nomiss_returns.append(total_r)
        if ep_acute:   acute_episodes += 1

    env_eval.close()
    env_clinical.close()

    print()
    print('=== Random Baseline: Clinical Environment (1000 episodes) ===')
    print(f'Overall mean return   : {np.mean(clinical_rand_returns):.4f}')
    print(f'Overall survival rate : {np.mean(np.array(clinical_rand_returns) > 0)*100:.1f}%')
    print()

    # Store for later comparison
    clinical_rand_mean = float(np.mean(clinical_rand_returns))

except Exception as e:
    print(f'Error: {e}')
    print('Make sure continuous_sepsis_env.py is in the project root and dependencies are installed.')
    clinical_rand_mean = 0.78


In [ ]:
test_env = make_clinical_env()
e = test_env
while hasattr(e, 'env'):
    print(type(e).__name__)
    e = e.env
print(type(e).__name__)
test_env.close()

In [ ]:
# Config B algorithms will be implemented here
### Your code here (Config B)
